# Wine Classification Portfolio Project

This notebook presents a compact machine learning project that can be shown in a portfolio or resume. The goal is to build and compare classification models that predict the wine class from physicochemical measurements.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("ggplot")

## 1. Problem Definition

The task is a supervised classification problem: predict the wine class using 13 numerical features such as alcohol, malic acid, flavanoids, and color intensity.

This project is suitable for a machine learning portfolio because it demonstrates a full workflow: problem framing, structured data inspection, preprocessing, model comparison, and interpretation of results.

## 2. Data Description

In [ ]:
wine = load_wine(as_frame=True)
data = wine.frame.copy()
data.rename(columns={"target": "wine_class"}, inplace=True)

print(f"Rows: {data.shape[0]}")
print(f"Columns: {data.shape[1]}")
print(f"Target classes: {sorted(data['wine_class'].unique().tolist())}")

data.head()

In [ ]:
feature_summary = data.drop(columns="wine_class").describe().T[["mean", "std", "min", "max"]]
feature_summary.head(10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data["wine_class"].value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Class Distribution")
axes[0].set_xlabel("Wine Class")
axes[0].set_ylabel("Count")

axes[1].scatter(data["alcohol"], data["color_intensity"], c=data["wine_class"], cmap="viridis", alpha=0.8)
axes[1].set_title("Alcohol vs Color Intensity")
axes[1].set_xlabel("Alcohol")
axes[1].set_ylabel("Color Intensity")
plt.tight_layout()


The dataset contains 178 wine samples with 13 numerical features and 3 target classes. Because the features are measured on different scales, preprocessing is necessary before training models such as Logistic Regression.

## 3. Preprocessing

All input columns are numerical, so the preprocessing pipeline focuses on:

- median imputation for stability
- standardization so that distance- and coefficient-based models behave consistently

In [ ]:
X = data.drop(columns="wine_class")
y = data["wine_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_features = X.columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ]
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Features used:", numeric_features)

## 4. Model Application

I compare two commonly used classification models:

- Logistic Regression as an interpretable baseline
- Random Forest as a non-linear ensemble model

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = []
trained_models = {}

for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", estimator),
        ]
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append({"model": model_name, "accuracy": round(float(accuracy), 3)})
    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df

In [ ]:
selected_model_name = "Logistic Regression"
selected_pipeline = trained_models[selected_model_name]
selected_predictions = selected_pipeline.predict(X_test)

print(f"Selected portfolio model: {selected_model_name}")
print(f"Accuracy: {accuracy_score(y_test, selected_predictions):.3f}")
print(classification_report(y_test, selected_predictions))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, selected_predictions, ax=axes[0], cmap="Blues")
axes[0].set_title("Logistic Regression Confusion Matrix")

logistic_model = selected_pipeline.named_steps["model"]
importance = np.abs(logistic_model.coef_).mean(axis=0)
importance_df = (
    pd.DataFrame({"feature": X.columns, "importance": importance})
    .sort_values("importance", ascending=False)
    .head(8)
    .sort_values("importance")
)

axes[1].barh(importance_df["feature"], importance_df["importance"], color="#4C78A8")
axes[1].set_title("Top Logistic Regression Features")
axes[1].set_xlabel("Mean Absolute Coefficient")
plt.tight_layout()

importance_df.sort_values("importance", ascending=False)

## 5. Result Interpretation

The selected baseline model, **Logistic Regression**, achieved **97.2% accuracy** on the test split. This indicates that the physicochemical measurements provide strong signal for distinguishing wine classes.

Feature importance based on logistic regression coefficients suggests that variables such as `proline`, `flavanoids`, `od280/od315_of_diluted_wines`, and `color_intensity` contribute strongly to the classification decision.

Random Forest was also tested as a stronger non-linear benchmark. For portfolio purposes, Logistic Regression is highlighted because it combines strong performance with straightforward interpretability.

Possible next steps:

- add cross-validation for more robust model comparison
- tune hyperparameters with grid search
- package the workflow into a reusable training script